In [15]:
import numpy as np
import pandas as pd
import xgboost as xgb
from pandas import concat
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import cross_val_predict
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import QuantileTransformer
import warnings
import matplotlib
from scipy.stats import ks_2samp
#from torch.distributed.pipelining import pipeline

matplotlib.use('inline')
# matplotlib.use('TkAgg')
from xgboost import XGBClassifier

In [16]:
# 解除警告
warnings.filterwarnings('ignore')
# matplotlib_inline.backend_inline.set_matplotlib_formats('svg')

In [17]:
# 读入数据
def read_data():

    # 读取online训练集
    online_train = pd.read_csv('ccf_online_stage1_train.csv', parse_dates=['Date', 'Date_received'])

    # 读取offline训练集
    offline_train = pd.read_csv('ccf_offline_stage1_train.csv', parse_dates=['Date', 'Date_received'])

    # 读取offline测试集
    offline_test = pd.read_csv('ccf_offline_stage1_test_revised.csv', parse_dates=['Date_received'])

    return online_train, offline_train, offline_test

on_train, off_train, off_test = read_data()

In [18]:
# offline预处理
def off_prior_processing(df):

    # 拷贝数据集
    df_processed = df.copy()

    # 用0标记优惠劵缺失值
    df_processed['Coupon_id'] = df_processed['Coupon_id'].fillna(0)
    df_processed['Coupon_id'] = df_processed['Coupon_id'].astype(int)

    # 将满减转换为折扣率
    df_processed[['discount_x', 'discount_y']] = df_processed[df_processed['Discount_rate'].str.contains(':') == True]['Discount_rate'].str.split(':', expand=True).astype(int)
    df_processed['discount_rate'] = 1 - (df_processed['discount_y'] / df_processed['discount_x'])
    df_processed['discount_rate'] = df_processed['discount_rate'].fillna(df_processed['Discount_rate']).astype(float)

    return df_processed

off_train_processed = off_prior_processing(off_train)
off_test_processed = off_prior_processing(off_test)
off_train_processed

,User_id,Merchant_id,Coupon_id,Discount_rate,Distance,Date_received,Date,discount_x,discount_y,discount_rate
0,1439408,2632,0,NaN,0.0,NaT,2016-02-17,NaN,NaN,NaN
1,1439408,4663,11002,150:20,1.0,2016-05-28,NaT,150.0,20.0,0.866667
2,1439408,2632,8591,20:1,0.0,2016-02-17,NaT,20.0,1.0,0.950000
3,1439408,2632,1078,20:1,0.0,2016-03-19,NaT,20.0,1.0,0.950000
4,1439408,2632,8591,20:1,0.0,2016-06-13,NaT,20.0,1.0,0.950000
...,...,...,...,...,...,...,...,...,...,...
1754879,212662,3532,0,NaN,1.0,NaT,2016-03-22,NaN,NaN,NaN
1754880,212662,3021,3739,30:1,6.0,2016-05-08,2016-06-02,30.0,1.0,0.966667
1754881,212662,2934,0,NaN,2.0,NaT,2016-03-21,NaN,NaN,NaN
1754882,752472,7113,1633,50:10,6.0,2016-06-13,NaT,50.0,10.0,0.800000


In [19]:
# online预处理
def on_prior_processing(df):

    # 拷贝数据
    df_processed = df.copy()

    # 标记优惠劵缺失值
    df_processed['Coupon_id'] = df_processed['Coupon_id'].fillna(0)

    return df_processed

on_train_processed = on_prior_processing(on_train)
on_train_processed

,User_id,Merchant_id,Action,Coupon_id,Discount_rate,Date_received,Date
0,13740231,18907,2,100017492,500:50,2016-05-13,NaT
1,13740231,34805,1,0,NaN,NaT,2016-03-21
2,14336199,18907,0,0,NaN,NaT,2016-06-18
3,14336199,18907,0,0,NaN,NaT,2016-06-18
4,14336199,18907,0,0,NaN,NaT,2016-06-18
...,...,...,...,...,...,...,...
11429821,13087731,27715,0,0,NaN,NaT,2016-06-29
11429822,13087731,52005,0,0,NaN,NaT,2016-03-24
11429823,13087731,45611,0,0,NaN,NaT,2016-04-22
11429824,13683699,18009,1,0,NaN,NaT,2016-03-23


In [20]:
# 特征提取
def get_features(val_data, off_data, on_data, test=False):

    """ 特征提取与处理 """

    # 创建新特征列表
    add_features = []
    """ 用户历史优惠券使用率 """

    # 计算用户在历史数据中的优惠券使用率
    if not test:
        # 训练阶段：使用提供的off_data计算历史使用率
        user_coupon_stats = off_data[off_data['Coupon_id'] != 0].groupby('User_id').apply(
            lambda x: pd.Series({
                'total_received': len(x),
                'total_used': len(x[(x['Date'].notna()) & ((x['Date'] - x['Date_received']).dt.days <= 15)])
            })
        ).reset_index()
    else:
        # 测试阶段：使用完整的训练数据计算历史使用率
        user_coupon_stats = off_train_processed[off_train_processed['Coupon_id'] != 0].groupby('User_id').apply(
            lambda x: pd.Series({
                'total_received': len(x),
                'total_used': len(x[(x['Date'].notna()) & ((x['Date'] - x['Date_received']).dt.days <= 15)])
            })
        ).reset_index()

    user_coupon_stats['user_coupon_usage_rate'] = user_coupon_stats['total_used'] / user_coupon_stats['total_received']
    user_coupon_stats['user_coupon_usage_rate'] = user_coupon_stats['user_coupon_usage_rate'].fillna(0)

    # 合并到当前数据集
    val_data = val_data.merge(user_coupon_stats[['User_id', 'user_coupon_usage_rate']].drop_duplicates(subset=['User_id']),on='User_id',how='left')
    val_data['user_coupon_usage_rate'] = val_data['user_coupon_usage_rate'].fillna(0)
    add_features.append('user_coupon_usage_rate')

    # 商家相关的特征--优惠券使用率
    if not test:
        merchant_stats = off_data.groupby('Merchant_id').apply(
        lambda x: pd.Series({
            'merchant_coupon_usage_rate': len(x[(x['Date'].notna()) & (x['Coupon_id'] != 0)]) / max(1, len(x[x['Coupon_id'] != 0])),
            'merchant_total_transactions': len(x),
            'merchant_coupon_transactions': len(x[x['Coupon_id'] != 0])
        })
    ).reset_index()
    else:
        merchant_stats = off_train_processed.groupby('Merchant_id').apply(
        lambda x: pd.Series({
            'merchant_coupon_usage_rate': len(x[(x['Date'].notna()) & (x['Coupon_id'] != 0)]) / max(1, len(x[x['Coupon_id'] != 0])),
            'merchant_total_transactions': len(x),
            'merchant_coupon_transactions': len(x[x['Coupon_id'] != 0])
        })
    ).reset_index()

    val_data = val_data.merge(merchant_stats[['Merchant_id', 'merchant_coupon_usage_rate']].drop_duplicates(subset=['Merchant_id']), on='Merchant_id', how='left')
    val_data['merchant_coupon_usage_rate'] = val_data['merchant_coupon_usage_rate'].fillna(0)
    add_features.append('merchant_coupon_usage_rate')

    # 1. 用户在该商户的历史交易次数
    if not test:
        user_merchant_transactions = off_data.groupby(['User_id', 'Merchant_id']).size().reset_index(name='user_merchant_transaction_count')
    else:
        user_merchant_transactions = off_train_processed.groupby(['User_id', 'Merchant_id']).size().reset_index(name='user_merchant_transaction_count')

    val_data = val_data.merge(user_merchant_transactions.drop_duplicates(subset=['User_id', 'Merchant_id']), on=['User_id', 'Merchant_id'], how='left')
    val_data['user_merchant_transaction_count'] = val_data['user_merchant_transaction_count'].fillna(0)
    add_features.append('user_merchant_transaction_count')

    """ offline - 原生特征 """

    # 用户与商家距离
    # temp = off_data['Distance'].median()
    # off_data['Distance'] = off_data['Distance'].fillna(temp)
    # val_data['Distance'] = val_data['Distance'].fillna(temp)
    add_features.append('Distance')

    # 满减卷x与y
    # temp_x = off_data['discount_x'].median()
    # temp_y = off_data['discount_y'].median()
    # off_data['discount_x'] = off_data['discount_x'].fillna(temp_x)
    # off_data['discount_y'] = off_data['discount_y'].fillna(temp_y)
    # val_data['discount_x'] = val_data['discount_x'].fillna(temp_x)
    # val_data['discount_y'] = val_data['discount_y'].fillna(temp_y)
    add_features.append('discount_x')
    add_features.append('discount_y')

    # 优惠劵力度
    # temp = off_data['discount_rate'].median()
    # off_data['discount_rate'] = off_data['discount_rate'].fillna(temp)
    # val_data['discount_rate'] = val_data['discount_rate'].fillna(temp)

    """ test - 其他特征 """

    # 用户在之后获取同类型优惠劵张数
    val_data['num_same_coupon_behind'] = val_data.groupby(['User_id', 'Discount_rate'])['Date_received'].rank(method='dense', ascending=False).astype(int) - 1
    add_features.append('num_same_coupon_behind')

    # 用户在之后获取所有优惠劵总数
    val_data['num_all_coupon_behind'] = val_data.groupby('User_id')['Date_received'].rank(method='dense', ascending=False).astype(int) - 1
    add_features.append('num_all_coupon_behind')

    # 用户对同类优惠卷获取的总数
    val_data['num_same_coupon'] = val_data.groupby(['User_id', 'Discount_rate'])['Discount_rate'].transform('count')
    add_features.append('num_same_coupon')

    # 用户领取优惠劵种类数 (删了掉分)
    val_data['coupon_sorts_num'] = val_data.groupby('User_id')['Discount_rate'].transform('nunique')
    add_features.append('coupon_sorts_num')

     # 商家在当前特征时间窗口内发放的优惠券总数
    merchant_coupon_count = off_data[off_data['Coupon_id'] != 0].groupby('Merchant_id').size().reset_index(name='merchant_total_coupons_issued')
    val_data = val_data.merge(merchant_coupon_count.drop_duplicates(subset='Merchant_id'), on='Merchant_id', how='left')
    val_data['merchant_total_coupons_issued'] = val_data['merchant_total_coupons_issued'].fillna(0)
    add_features.append('merchant_total_coupons_issued')




    #删去多余列
    val_data = val_data.drop(['User_id', 'Merchant_id', 'Discount_rate','Coupon_id', 'Date_received'], axis=1)

    if test:
        return val_data, add_features
    else:
        val_data = val_data.drop('Date', axis=1)
        return val_data

In [21]:
#时间线处理
def dataset_split(on_train_, off_train_, off_test_):

    """ 时间分割与数据集筛选 """

    """ 第一训练集 """

    #验证集筛选
    time1 = ['2016-04-16', '2016-05-15']
    off_val1 = off_train_[(off_train_['Date_received'] >= pd.to_datetime(time1[0])) & (off_train_['Date_received'] <= pd.to_datetime(time1[1]))]
    off_val1['label'] = 0
    off_val1.loc[(off_val1['Date']).notna() & ((off_val1['Date'] - off_val1['Date_received']).dt.days <= 15), 'label'] = 1

    #特征集筛选
    date1 = ['2016-01-01', '2016-04-15']
    features1_off = off_train_[(off_train_['Date'] >= pd.to_datetime(date1[0])) & (off_train_['Date'] <= pd.to_datetime(date1[1])) |
                              ((off_train_['Date']).isna() & (off_train_['Date_received'] >= pd.to_datetime(date1[0]))) &
                              (off_train_['Date_received'] <= pd.to_datetime(date1[1]))]
    features1_on = on_train_[(on_train_['Date'] >= pd.to_datetime(date1[0])) & (on_train_['Date'] <= pd.to_datetime(date1[1])) |
                              ((on_train_['Date']).isna() & (on_train_['Date_received'] >= pd.to_datetime(date1[0]))) &
                              (on_train_['Date_received'] <= pd.to_datetime(date1[1]))]

    """ 第二训练集 """

    #验证集筛选
    time2 = ['2016-05-16', '2016-06-15']
    off_val2 = off_train_[(off_train_['Date_received'] >= pd.to_datetime(time2[0])) & (off_train_['Date_received'] <= pd.to_datetime(time2[1]))]
    off_val2['label'] = 0
    off_val2.loc[(off_val2['Date']).notna() & ((off_val2['Date'] - off_val2['Date_received']).dt.days <= 15), 'label'] = 1

    #特征集筛选
    date2 = ['2016-02-01', '2016-05-15']
    features2_off = off_train_[(off_train_['Date'] >= pd.to_datetime(date2[0])) & (off_train_['Date'] <= pd.to_datetime(date2[1])) |
                              ((off_train_['Date']).isna() & (off_train_['Date_received'] >= pd.to_datetime(date2[0]))) &
                              (off_train_['Date_received'] <= pd.to_datetime(date2[1]))]
    features2_on = on_train_[(on_train_['Date'] >= pd.to_datetime(date2[0])) & (on_train_['Date'] <= pd.to_datetime(date2[1])) |
                              ((on_train_['Date']).isna() & (on_train_['Date_received'] >= pd.to_datetime(date2[0]))) &
                              (on_train_['Date_received'] <= pd.to_datetime(date2[1]))]

    """ 测试集 """

    #测试集筛选
    off_test3 = off_test_

    #特征集筛选
    date3 = ['2016-03-16', '2016-06-30']
    features3_off = off_train_[(off_train_['Date'] >= pd.to_datetime(date3[0])) & (off_train_['Date'] <= pd.to_datetime(date3[1])) |
                              ((off_train_['Date']).isna() & (off_train_['Date_received'] >= pd.to_datetime(date3[0]))) &
                              (off_train_['Date_received'] <= pd.to_datetime(date3[1]))]
    features3_on = on_train_[(on_train_['Date'] >= pd.to_datetime(date3[0])) & (on_train_['Date'] <= pd.to_datetime(date3[1])) |
                              ((on_train_['Date']).isna() & (on_train_['Date_received'] >= pd.to_datetime(date3[0]))) &
                              (on_train_['Date_received'] <= pd.to_datetime(date3[1]))]

    """ 数据拼接 """

    f_val1 = get_features(off_val1, features1_off, features1_on)
    f_val2 = get_features(off_val2, features2_off, features2_on)
    f_test3, features_list = get_features(off_test3, features3_off, features3_on, test=True)

    """ 最终训练集 """
    # f_train = pd.concat([off_val1, off_val2], axis=0)
    # f_features_off = pd.concat([features1_off, features2_off], axis=0)
    # f_features_on = pd.concat([features1_on, features2_on], axis=0)
    # f_all = get_features(f_train, f_features_off, f_features_on)

    return f_val1, f_val2, f_test3, features_list

In [22]:
#接受数据集
F_val1, F_val2, F_test3, F_list = dataset_split(on_train_processed, off_train_processed, off_test_processed)

In [23]:
F_all = pd.concat([F_val1, F_val2], axis=0)

In [24]:
#特征与标签分离
def x_y_split(df):
    x = df.drop('label', axis=1)
    y = df['label']
    return x, y

x_val1, y_val1 = x_y_split(F_val1)
x_val2, y_val2 = x_y_split(F_val2)
x_all, y_all = x_y_split(F_all)
x_all_copy = x_all.copy()
F_test3_copy = F_test3.copy()
x_all

,Distance,discount_x,discount_y,discount_rate,user_coupon_usage_rate,merchant_coupon_usage_rate,user_merchant_transaction_count,num_same_coupon_behind,num_all_coupon_behind,num_same_coupon,coupon_sorts_num,merchant_total_coupons_issued
0,0.0,200.0,20.0,0.900000,0.0,0.021246,0.0,0,0,1,1,103033.0
1,10.0,200.0,30.0,0.850000,0.0,0.004636,0.0,0,0,1,1,22863.0
2,0.0,20.0,1.0,0.950000,0.0,0.000000,0.0,0,0,1,1,0.0
3,10.0,200.0,20.0,0.900000,0.0,0.021246,1.0,0,0,1,1,103033.0
4,NaN,30.0,5.0,0.833333,0.0,0.012633,0.0,0,0,1,1,32691.0
...,...,...,...,...,...,...,...,...,...,...,...,...
252581,0.0,30.0,5.0,0.833333,0.0,0.122348,0.0,0,0,1,1,6081.0
252582,0.0,30.0,5.0,0.833333,0.0,0.086504,1.0,0,0,1,1,4705.0
252583,NaN,100.0,30.0,0.700000,0.0,0.066667,0.0,0,0,1,1,15.0
252584,6.0,50.0,10.0,0.800000,0.0,0.027914,0.0,0,0,1,2,2042.0


In [25]:
F_test3

,Distance,discount_x,discount_y,discount_rate,user_coupon_usage_rate,merchant_coupon_usage_rate,user_merchant_transaction_count,num_same_coupon_behind,num_all_coupon_behind,num_same_coupon,coupon_sorts_num,merchant_total_coupons_issued
0,1.0,30.0,5.0,0.833333,0.000000,0.023175,1.0,0,0,1,1,35931.0
1,NaN,30.0,5.0,0.833333,1.000000,0.170000,0.0,0,0,1,1,100.0
2,5.0,200.0,20.0,0.900000,0.000000,0.021809,0.0,0,0,1,2,3503.0
3,5.0,100.0,10.0,0.900000,0.000000,0.021809,0.0,0,0,1,2,3503.0
4,2.0,30.0,1.0,0.966667,0.000000,0.228931,0.0,0,0,1,1,795.0
...,...,...,...,...,...,...,...,...,...,...,...,...
113635,10.0,30.0,5.0,0.833333,0.000000,0.022835,1.0,0,0,1,2,7759.0
113636,NaN,30.0,1.0,0.966667,0.333333,0.000000,0.0,0,1,1,2,8.0
113637,NaN,50.0,5.0,0.900000,0.333333,0.043478,0.0,0,0,1,2,23.0
113638,0.0,30.0,5.0,0.833333,0.250000,0.069815,1.0,0,0,1,1,34110.0


In [26]:
params = {
    'booster': 'gbtree',
              'objective': 'binary:logistic',
              'eval_metric': 'auc',
              'silent': 1,
              'eta': 0.01,
              'max_depth': 5,
              'min_child_weight': 1,
              'gamma': 0,
              'lambda': 1,
              'colsample_bylevel': 0.7,
              'colsample_bytree': 0.7,
              'subsample': 0.9,
              'scale_pos_weight': 1
}

In [27]:
clf = xgb.XGBClassifier(**params)

In [28]:
clf.fit(x_all, y_all)
F_predicted = clf.predict_proba(F_test3)
def get_submission(test, pred):
    pred = pred[:, 1]
    pred = pd.DataFrame(pred, columns=['pred'])
    submission_ = pd.concat([test, pred], axis=1)
    submission_ = submission_[['User_id', 'Coupon_id', 'Date_received', 'pred']]
    submission_['User_id'] = submission_['User_id'].astype(int)
    submission_['Coupon_id'] = submission_['Coupon_id'].astype(int)
    submission_['Date_received'] = submission_['Date_received'].dt.strftime('%Y%m%d').astype(int)
    return submission_
submission = get_submission(off_test, F_predicted)
submission.to_csv('submission1.csv', index=False, header=False)